In [ ]:
# 读机器人模型文件（URDF）：先找到文件，再解析
import json                                    # 把读出的结果存成文件，课件里的数从这里取
import math                                    # math.degrees 把弧度换成度
import warnings                                # 控制第三方库的提示
import xml.etree.ElementTree as ET             # URDF 是 XML 文件，标准库就能解析
from pathlib import Path                       # 拼文件路径

warnings.filterwarnings("ignore")              # 关掉第三方库的提示，让输出干净
import so101_sim                               # 星禾仿真包：机器人模型文件就放在它里面

urdf_path = Path(so101_sim.__file__).parent / "robots" / "kit_assets" / "kit_v1_so101.urdf"   # 模型文件的路径
root = ET.parse(urdf_path).getroot()           # 解析成一棵树，根节点是 <robot>
print(urdf_path.name, "里有", len(root.findall("link")), "个连杆、", len(root.findall("joint")), "个关节")   # 先数一数

In [ ]:
# 把每个关节的名字、类型、父子连杆、位置、转轴和上下限取出来
rows = []                                                  # 每个关节存成一行
for joint in root.findall("joint"):                        # 逐个 <joint> 处理
    row = {
        "name": joint.get("name"),                         # 关节名，如 shoulder_pan
        "type": joint.get("type"),                         # 类型：revolute 是转动关节，fixed 是固定关节
        "parent": joint.find("parent").get("link"),        # 父连杆：这个关节接在哪个连杆上
        "child": joint.find("child").get("link"),          # 子连杆：这个关节带着哪个连杆转
        "origin_m": [float(v) for v in joint.find("origin").get("xyz").split()],   # 关节原点在父连杆坐标系里的位置，单位是米
    }
    if row["type"] == "revolute":                          # 只有转动关节有转轴和上下限
        row["axis"] = [float(v) for v in joint.find("axis").get("xyz").split()]   # 转轴方向，写在关节自己的坐标系里
        row["lower_deg"] = math.degrees(float(joint.find("limit").get("lower")))  # 下限：弧度换成度
        row["upper_deg"] = math.degrees(float(joint.find("limit").get("upper")))  # 上限：弧度换成度
    rows.append(row)                                       # 存进列表

In [ ]:
# 只看活动关节（转动关节），打印成一张表
active = [r for r in rows if r["type"] == "revolute"]      # 固定关节不会动，活动关节才有自由度
print(f'{"joint":<15}{"type":<10}{"parent":<15}{"child":<26}{"axis":<18}{"lower":>8}{"upper":>8}')   # 表头，上下限的单位是度
for r in active:                                           # 每个活动关节打印一行
    print(f'{r["name"]:<15}{r["type"]:<10}{r["parent"]:<15}{r["child"]:<26}{str(r["axis"]):<18}{r["lower_deg"]:>8.1f}{r["upper_deg"]:>8.1f}')
print("活动关节", len(active), "个，固定关节", len(rows) - len(active), "个")   # 数一数活动关节有几个

In [ ]:
# 课程采用的限位：前四个臂关节取模型文件上下限里较小的一侧，写成对称的范围；腕滚转与夹爪的零位按安装标定，范围本来就不对称，照模型文件
symmetric = ["shoulder_pan", "shoulder_lift", "elbow_flex", "wrist_flex"]   # 写成对称范围的四个关节
course = {}                                                # 关节名 → (下限, 上限)，单位是度
for r in active:                                           # 逐个活动关节处理
    if r["name"] in symmetric:                             # 前四个臂关节
        half = min(-r["lower_deg"], r["upper_deg"])        # 上下限里离零较近的一侧
        course[r["name"]] = (-half, half)                  # 写成对称的范围
    else:                                                  # 腕滚转与夹爪
        course[r["name"]] = (r["lower_deg"], r["upper_deg"])   # 照模型文件
for name, (lower, upper) in course.items():                # 打印出来看一眼
    print(f"{name:<15}{lower:8.1f}{upper:8.1f}")           # 下限与上限，单位是度

In [ ]:
# 每个连杆的质量：URDF 里 <inertial> 写着，单位是千克
masses = {}                                                # 连杆名 → 质量
for link in root.findall("link"):                          # 逐个 <link> 处理
    mass = link.find("inertial/mass")                      # 有质量属性的连杆才有这一项
    if mass is not None and float(mass.get("value")) > 1e-6:   # 相机坐标系、指尖这类只是参考系，质量为零，不算
        masses[link.get("name")] = float(mass.get("value"))    # 记下这个连杆的质量
print("有质量的连杆", len(masses), "个，合计", round(sum(masses.values()), 3), "kg")   # 整条臂有多重

In [ ]:
# 存成文件：课件里的限位、连杆长度和质量从这里读（限位存课程采用的那一份）
out = Path("out") / "03-urdf-joints.json"                  # 输出文件，放在 out 目录里
out.parent.mkdir(exist_ok=True)                            # out 目录不存在就建一个
out.write_text(json.dumps({"n_active": len(active), "joints": rows, "masses": masses, "course": course}, ensure_ascii=False, indent=1))   # 写成 JSON
print("已存到", out)                                       # 告诉自己存在哪里

In [ ]:
# 建仿真环境：动作用绝对位置，也就是每个关节的目标角度
import gymnasium as gym                          # 仿真环境的标准接口
import numpy as np                               # 数组运算
import matplotlib.pyplot as plt                  # 画图，用来拼接触印样
from mani_skill.utils import sapien_utils        # 摆相机位置用的小工具
import so101_sim                                 # 导入它，星禾仿真的场景就注册好了

camera = dict(pose=sapien_utils.look_at([0.42, 0.38, 0.55], [0.08, 0.0, 0.08]), width=640, height=480, fov=np.deg2rad(50))   # 一台从斜前方偏高处看机械臂的相机
env = gym.make("SO101PickPlaceCube40-v1", obs_mode="rgb", control_mode="pd_joint_pos", render_mode="rgb_array", human_render_camera_configs=camera)   # 方块放进料盒的场景，绝对位置控制
obs, _ = env.reset(seed=1)                       # 复位，固定随机种子 1
names = [j.name for j in env.unwrapped.agent.robot.active_joints]   # 仿真里关节的顺序，动作向量按这个顺序排
print(names)                                     # 和 URDF 里的关节名对一对

In [ ]:
# 逐个转动五个臂关节：先转到下限附近，再转到上限附近，每个位置拍一张画面
margin = 5                                       # 离限位留 5°，不去撞限位
home = obs["agent"]["noisy_qpos"][0].cpu().numpy()   # 每个关节当前的角度（弧度）：复位后的起始姿势
print("起始姿势（度）：", np.rad2deg(home).round(1))  # 换成度打印：五个臂关节加夹爪
limits = course                                  # 上一步得出的课程采用的限位，单位是度
shots = []                                       # 存每个位置的画面和标题
for i, name in enumerate(names[:5]):             # 五个臂关节，夹爪不转
    lower, upper = limits[name]                  # 这个关节的下限、上限
    for target in (lower + margin, upper - margin):   # 先到下限附近，再到上限附近
        action = home.copy()                     # 其余关节保持起始角度
        action[i] = np.deg2rad(target)           # 这个关节的目标角度：度换成弧度
        for _ in range(60):                      # 发 60 步，30 Hz 下是 2 秒，够转到位
            obs, *_ = env.step(action)           # 仿真走一步
        shots.append((env.render()[0].cpu().numpy(), f"{name} {target:.0f}°"))   # 拍一张画面

In [ ]:
# 把十张画面拼成一张接触印样：上一行是下限附近，下一行是上限附近
fig, axes = plt.subplots(2, 5, figsize=(20, 6.5))   # 两行五列
for k, (image, title) in enumerate(shots):       # 逐张摆进去
    ax = axes[k % 2][k // 2]                     # 偶数张是下限（上一行），奇数张是上限（下一行）
    ax.imshow(image)                             # 画面
    ax.set_title(title)                          # 标题：关节名和目标角度
    ax.axis("off")                               # 不画坐标轴
fig.tight_layout()                               # 排紧一点
fig.savefig("out/03-joint-sweep.png", dpi=100)   # 存盘，课件里用得上
plt.show()                                       # 在 notebook 里显示

In [ ]:
# 越过限位会怎样：目标定在限位之外 20°，发一段时间，读回关节停在哪里
tests = [("shoulder_pan", "upper"), ("elbow_flex", "upper"), ("wrist_flex", "lower"), ("wrist_roll", "upper")]   # 每个关节试一个方向；肩俯仰朝上限转会先碰到桌面，这里不试
beyond = []                                      # 每个关节存一行
for name, side in tests:                         # 逐个关节试
    obs, _ = env.reset(seed=1)                   # 每次都从复位姿势出发
    i = names.index(name)                        # 这个关节在动作向量里的位置
    edge = limits[name][1] if side == "upper" else limits[name][0]   # 这一侧的限位（度）
    target = edge + 20 if side == "upper" else edge - 20             # 目标：限位之外 20°
    action = home.copy()                         # 其余关节保持起始角度
    action[i] = np.deg2rad(target)               # 这个关节的目标角度：度换成弧度
    for _ in range(90):                          # 发 90 步，足够转到停住
        obs, *_ = env.step(action)               # 仿真走一步
    read = float(np.rad2deg(obs["agent"]["noisy_qpos"][0].cpu().numpy())[i])   # 读回这个关节现在的角度（度）
    beyond.append({"joint": name, "side": side, "target_deg": round(target, 1), "read_deg": round(read, 1), "limit_deg": round(edge, 1)})   # 记下这一行
    print(f"{name:<15}目标 {target:7.1f}°   读回 {read:7.1f}°   限位 {edge:7.1f}°")   # 目标、读回、限位三个数对照着看

In [ ]:
# 关节空间到末端空间：取三组关节角，发给仿真，读夹爪中心的位置
rng = np.random.default_rng(3)                   # 固定随机种子，每次取到同样的三组
poses = []                                       # 每组存关节角和夹爪中心位置
for _ in range(3):                               # 取三组
    q_deg = [rng.uniform(limits[n][0] + margin, limits[n][1] - margin) for n in names[:5]]   # 五个关节各在限位内随机取一个角度
    action = np.append(np.deg2rad(q_deg), home[5])   # 加上夹爪，保持起始开度
    for _ in range(90):                          # 发 90 步，等机械臂停稳
        obs, *_ = env.step(action)               # 仿真走一步
    q_read = np.rad2deg(obs["agent"]["noisy_qpos"][0].cpu().numpy())[:5]   # 读回五个关节角，弧度换成度
    tcp = env.unwrapped.agent.tcp_pos[0].cpu().numpy()   # 仿真给出的夹爪中心位置：真值，单位是米
    poses.append({"q_deg": q_read.astype(float).round(1).tolist(), "tcp_m": tcp.astype(float).round(3).tolist()})   # 记下这一组

In [ ]:
# 打印对照表：左边是关节空间的五个角度，右边是末端空间的位置
for p in poses:                                  # 三组，一组一行
    print("q =", p["q_deg"], "°   →   p =", p["tcp_m"], "m")   # 五个关节角，对应夹爪中心的位置

In [ ]:
# 工具中心点是两指尖的中点，不是末端连杆的坐标原点：量一量相距多少，再量夹爪闭合与全开时两指尖相距多少
agent = env.unwrapped.agent                      # 仿真里的机器人对象
tcp = agent.tcp_pos[0].cpu().numpy()             # 工具中心点的位置，单位是米
origin = agent.robot.links_map["gripper_link"].pose.p[0].cpu().numpy()   # 末端连杆坐标原点的位置，单位是米
offset_cm = float(np.linalg.norm(tcp - origin)) * 100   # 两点相距多少厘米
gap_cm = {}                                      # 夹爪开度（百分比）→ 两指尖相距（厘米）
for pct in (0, 100):                             # 闭合和全开两种
    action = home.copy()                         # 五个臂关节保持起始角度
    action[5] = np.deg2rad(-10 + 1.1 * pct)      # 夹爪角度：−10° 到 100° 对应 0 到 100%
    for _ in range(60):                          # 发 60 步，等夹爪到位
        obs, *_ = env.step(action)               # 仿真走一步
    tips = agent.finger1_tip.pose.p[0].cpu().numpy() - agent.finger2_tip.pose.p[0].cpu().numpy()   # 两个指尖位置的差
    gap_cm[pct] = round(float(np.linalg.norm(tips)) * 100, 1)   # 距离，换成厘米
print(f"工具中心点离末端连杆原点 {offset_cm:.1f} cm；两指尖相距：闭合 {gap_cm[0]} cm，全开 {gap_cm[100]} cm")   # 打印

In [ ]:
# 存成文件，课件里的数从这里读
Path("out/03-joint-vs-tcp.json").write_text(json.dumps(poses, ensure_ascii=False, indent=1))   # 三组关节角与夹爪中心位置
Path("out/03-tcp-and-limits.json").write_text(json.dumps({"home_deg": np.rad2deg(home).round(1).tolist(), "tcp_offset_cm": round(offset_cm, 1), "gap_cm": gap_cm, "beyond": beyond}, ensure_ascii=False, indent=1))   # 起始姿势、工具中心点的偏移、指尖间距、越限的读回
print("已存到 out/03-joint-vs-tcp.json 与 out/03-tcp-and-limits.json")   # 告诉自己存在哪里

In [ ]:
# 每个关节单独多转 10°，夹爪中心在空间里挪了多远？先摆成臂向前伸平的姿势
reach_deg = [0, -45, 45, 0, 0]                   # 伸平姿势的五个臂关节角（度），课件里那张图用的就是它
grip = np.deg2rad(-10 + 1.1 * 70)                # 夹爪张开 70%
def settle(arm_deg):                             # 把五个臂关节送到给定角度，等稳，读夹爪中心的位置
    action = np.append(np.deg2rad(arm_deg), grip)   # 动作：五个臂关节加夹爪
    for _ in range(120):                         # 发 120 步，等机械臂停稳
        env.step(action)                         # 仿真走一步
    return env.unwrapped.agent.tcp_pos[0].cpu().numpy()   # 夹爪中心的位置，单位是米
start = settle(reach_deg)                        # 伸平姿势下的夹爪中心
moves = []                                       # 每个关节存一行
for i, name in enumerate(names[:5]):             # 五个臂关节
    turned = list(reach_deg)                     # 复制伸平姿势
    turned[i] += 10                              # 只有这个关节多转 10°
    moved = float(np.linalg.norm(settle(turned) - start)) * 100   # 夹爪中心挪动的距离，换成厘米
    moves.append({"joint": name, "move_cm": round(moved, 1)})     # 记下这个关节
    print(f"{name:<15}多转 10°，夹爪中心移动 {moved:5.1f} cm")      # 打印一行
    settle(reach_deg)                            # 回到伸平姿势，再测下一个关节

In [ ]:
# 画成条形图，存成文件：课件里的数从这里读
fig, ax = plt.subplots(figsize=(8, 3.5))         # 一张横向条形图
ax.barh([m["joint"] for m in moves][::-1], [m["move_cm"] for m in moves][::-1])   # 离底座近的关节画在上面
ax.set_xlabel("夹爪中心移动的距离 (cm)")           # 横轴的名字
fig.tight_layout()                               # 排紧一点
fig.savefig("out/03-joint-influence.png", dpi=100)   # 存盘
Path("out/03-joint-influence.json").write_text(json.dumps({"reach_deg": reach_deg, "step_deg": 10, "moves": moves}, ensure_ascii=False, indent=1))   # 数存成 JSON
plt.show()                                       # 在 notebook 里显示

In [ ]:
# 关掉仿真环境
env.close()                                      # 释放显存